In [1]:
!pip install -q langchain==0.1.16 langchain-openai openai chromadb gradio python-dotenv tiktoken langchain-community faiss-cpu
# !pip install -q --upgrade langchain langchain-openai openai chromadb gradio python-dotenv tiktoken langchain-community
print("Libraries installed successfully!")

Libraries installed successfully!


In [2]:
# !pip -q install --upgrade openai

In [3]:
from openai import OpenAI

import os
from dotenv import load_dotenv
load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")

# Configure the OpenAI Client
openai_client = OpenAI(api_key=openai_api_key)
print("OpenAI client successfully configured.")
print(openai_api_key[:15])

OpenAI client successfully configured.
sk-proj-hpBtYcf


In [4]:
# from langchain.embeddings import OpenAIEmbeddings
# from langchain.llms import OpenAI

from langchain_openai import OpenAIEmbeddings, OpenAI
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain.chains import RetrievalQAWithSourcesChain
# from langchain.chains.qa_with_sources import RetrievalQAWithSourcesChain
from langchain.prompts import PromptTemplate

# from langchain.chains import create_retrieval_chain
# from langchain.chains.combine_documents import create_stuff_documents_chain

from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings

In [5]:
DATA_FILE_PATH = "financial_documents.txt"
print(f"Loading data from: {DATA_FILE_PATH}")

loader = TextLoader(DATA_FILE_PATH, encoding="utf-8")
raw_documents = loader.load()

print(f"Loaded {len(raw_documents)} document(s)")

Loading data from: financial_documents.txt
Loaded 1 document(s)


In [6]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=50,
    chunk_overlap=2
)

documents = text_splitter.split_documents(raw_documents)

if not documents:
    raise ValueError("No documents after splitting")

print(f"Split into {len(documents)} chunks")

print("\n--- Example Chunk (Chunk 2) ---")
print(documents[2].page_content)
print("\n--- Metadata for Chunk 2 ---")
print(documents[2].metadata)

Split into 699 chunks

--- Example Chunk (Chunk 2) ---
SECTION 1: BUSINESS OVERVIEW

--- Metadata for Chunk 2 ---
{'source': 'financial_documents.txt'}


In [7]:
# documents = documents * 200
# print(f"Simulated corpus size: {len(documents)} chunks")

In [8]:
embeddings = OpenAIEmbeddings(openai_api_key=openai_api_key)

vector_store = Chroma.from_documents(
    documents=documents,
    embedding=embeddings
)

print(f"Vector store contains {vector_store._collection.count()} embeddings")

Vector store contains 699 embeddings


In [9]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

llm = OpenAI(temperature=0,openai_api_key=openai_api_key)

In [10]:
prompt_template = """
You are a document intelligence assistant for financial documents.

Answer the question using ONLY the provided context.
If the answer is not contained in the context, respond with:
"I don't know based on the provided documents."

Context:
{context}

Question:
{question}

Answer concisely and cite sources.
"""

PROMPT = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)


In [11]:
qa_chain = RetrievalQAWithSourcesChain.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    chain_type_kwargs={
        "prompt": PROMPT,
        "document_variable_name": "context"
    },
    return_source_documents=True,
    verbose=True
)

print("RAG pipeline initialized successfully")

RAG pipeline initialized successfully


In [12]:
# query = "What services are described in the documents?"
# query = "Which firm identifies cybersecurity as a material risk, and in what context?"
query = "Which companies face commercial real estate risk? and show the statement"
# query = "Which firms rely primarily on fee-based revenue? and show evidence"
result = qa_chain.invoke({"question": query})

print("\n--- Answer ---")
# print(result["answer"])
print(result.get("answer", "No answer generated."))

print("\n--- Sources ---")
# print(result["sources"])
print(result.get("sources", "No sources identified."))



> Entering new RetrievalQAWithSourcesChain chain...

> Finished chain.

--- Answer ---

I don't know based on the provided documents.

--- Sources ---



In [13]:
def evaluate_hallucinations(chain, eval_set):
    hallucinations = 0

    for item in eval_set:
        result = chain.invoke({"question": item["q"]})
        answer = result["answer"].lower()

        if not item["answerable"] and "i don't know" not in answer:
            hallucinations += 1

    hallucination_rate = hallucinations / len(eval_set)
    return hallucination_rate


evaluation_set = [
    {"q": "What services are described in the documents?", "answerable": True},
    {"q": "Does the organization accept cryptocurrency payments?", "answerable": False},
    {"q": "Is there mention of a quantum computing platform?", "answerable": False},
    {"q": "What factors influence a global investment bank's trading revenue?", "answerable": True},
    {"q": "How do market declines impact an asset management firm's revenue?", "answerable": True},
    {"q": "What is the main business of a credit card network company?", "answerable": True},
    {"q": "Which companies face commercial real estate risk?", "answerable": True},
    {"q": "Are consumer credit conditions improving or deteriorating?", "answerable": False},
    {"q": "Which firms rely primarily on fee-based revenue?", "answerable": True},
    {"q": "What factors contributed to margin pressure?", "answerable": True},
    {"q": "Which risks are forward-looking versus currently observed?", "answerable": False}
]



hallucination_rate = evaluate_hallucinations(qa_chain, evaluation_set)
print(f"Hallucination rate: {hallucination_rate:.2f}")
print(f"Hallucination reduction: {(1 - hallucination_rate) * 100:.1f}%")




> Entering new RetrievalQAWithSourcesChain chain...

> Finished chain.


> Entering new RetrievalQAWithSourcesChain chain...

> Finished chain.


> Entering new RetrievalQAWithSourcesChain chain...

> Finished chain.


> Entering new RetrievalQAWithSourcesChain chain...

> Finished chain.


> Entering new RetrievalQAWithSourcesChain chain...

> Finished chain.


> Entering new RetrievalQAWithSourcesChain chain...

> Finished chain.


> Entering new RetrievalQAWithSourcesChain chain...

> Finished chain.


> Entering new RetrievalQAWithSourcesChain chain...

> Finished chain.


> Entering new RetrievalQAWithSourcesChain chain...

> Finished chain.


> Entering new RetrievalQAWithSourcesChain chain...

> Finished chain.


> Entering new RetrievalQAWithSourcesChain chain...

> Finished chain.
Hallucination rate: 0.36
Hallucination reduction: 63.6%


# FAISS + Hugging Face vs OpenAI Embeddings Comparison

In [14]:
def build_rag_pipeline(embedding_model, documents):
    vector_store = FAISS.from_documents(
        documents=documents,
        embedding=embedding_model
    )

    retriever = vector_store.as_retriever(search_kwargs={"k": 3})

    llm = OpenAI(
        temperature=0,
        openai_api_key=openai_api_key
    )

    qa_chain = RetrievalQAWithSourcesChain.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=retriever,
        chain_type_kwargs={"prompt": PROMPT,
        "document_variable_name": "context"},
        return_source_documents=True,
        verbose=False
    )

    return qa_chain

In [15]:
# openai_embeddings = OpenAIEmbeddings(
#     openai_api_key=OPENAI_API_KEY
# )

rag_openai = build_rag_pipeline(
    embedding_model=embeddings,
    documents=documents
)

openai_hallucination = evaluate_hallucinations(
    rag_openai, evaluation_set
)

In [ ]:
hf_embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [17]:
rag_hf = build_rag_pipeline(
    embedding_model=hf_embeddings,
    documents=documents
)

hf_hallucination = evaluate_hallucinations(
    rag_hf, evaluation_set
)

In [18]:
print("\n===== RESULTS =====")

print(f"OpenAI Embeddings:")
print(f"  Hallucination rate: {openai_hallucination:.2f}")


print(f"\nHugging Face Embeddings:")
print(f"  Hallucination rate: {hf_hallucination:.2f}")


===== RESULTS =====
OpenAI Embeddings:
  Hallucination rate: 0.36

Hugging Face Embeddings:
  Hallucination rate: 0.27
